# 📖 Notebook 1: Intro to Load Balancing & Round Robin

Welcome! Before we talk about fancy algorithms, let's understand **what a load
balancer is**, **why we need one**, and watch a single-server setup fall over
under load. Then we'll fix it with the simplest possible strategy:
**round robin**.

## Learning Objectives

By the end of this notebook, you'll understand:

- What a load balancer does, in plain English
- Why a single server is a bottleneck *and* a single point of failure
- The difference between **bad** (one server) and **better** (round robin
  across many servers)
- How to simulate backends in pure Python so you can experiment safely


## 🛠️ Setup

From the lab folder:

```bash
cd 01-foundations/load-balancing
uv sync
```

### Kernel selection

In VS Code, click the kernel picker at the **top-right** of this notebook and
choose the `.venv` interpreter (it will be named something like
`.venv (Python 3.x)`).

If the `.venv` kernel doesn't appear in the list, reload the VS Code window:

- `Cmd+Shift+P` (macOS) or `Ctrl+Shift+P` (Windows/Linux)
- Type and select **"Developer: Reload Window"**

No Docker or external services are needed for this lab — everything runs
in-process with plain Python.


## 🤔 What is a load balancer?

Imagine a very popular coffee shop with **one** barista. No matter how fast
they work, at some point the line out the door is too long and customers walk
away. The fix is obvious: hire more baristas and have a **host at the door**
who sends each new customer to whichever barista is free.

That host at the door is a **load balancer**. In a web system:

- **Clients** = customers placing orders
- **Backends / servers** = baristas making drinks
- **Load balancer** = the host deciding who gets which customer

A load balancer sits between clients and servers and decides, for every
incoming request, *which backend should handle it*. Its jobs:

1. **Spread the work** so no single backend gets overwhelmed.
2. **Skip dead backends** — if a barista goes home sick, stop sending
   customers to them.
3. **Scale horizontally** — add more baristas (backends) when it gets busy.

> 💡 The load balancer itself must be fast and simple. If *it* becomes the
> bottleneck, you're back where you started.


## 🌐 L4 vs L7: at what layer does the LB decide?

You'll see two flavors of load balancers in the wild. The difference is **how
deep into the request the LB looks** before picking a backend:

| Layer | Looks at | Examples | Pros | Cons |
|---|---|---|---|---|
| **L4** (Transport) | IP address + TCP/UDP port — *not* the HTTP body | AWS NLB, HAProxy `mode tcp`, IPVS | Very fast, handles any protocol (gRPC, MQTT, plain TCP) | Can't make smart decisions based on URL or headers |
| **L7** (Application) | Full HTTP request: path, headers, cookies | NGINX, Envoy, AWS ALB, Traefik | Route by URL (`/api/*` → API pool), terminate TLS, set sticky cookies | Slightly slower; protocol-specific |

> 💡 Rule of thumb: if you need to route based on **URL or headers**, you need
> **L7**. If you just want to spread raw TCP connections across servers, **L4**
> is simpler and faster.

Everything we build in this lab is a **conceptual L7** load balancer — we look
at the whole `Request` object, not just an IP+port pair. The algorithms
themselves are the same at both layers.


## 🧪 Let's simulate a backend

We don't need real servers to learn load balancing. A backend is just
"something that takes a request and takes some time to handle it". We can
model that with a small Python class.

We'll use **pydantic** (v2) to describe a `Request` — this gives us
validation for free and a clean place to document the fields.


In [ ]:
from pydantic import BaseModel, Field
import time
import random

class Request(BaseModel):
    """A single incoming request we want some backend to handle."""
    id: int
    # How much "work" this request represents, in seconds.
    # In a real system this varies a lot (a search query vs. a file upload).
    work_seconds: float = Field(gt=0)


class Backend:
    """A pretend backend server.

    It tracks two things we care about for load balancing:
      - how many requests it has handled (for fairness checks)
      - total time it has spent working (for load-distribution checks)
    """

    def __init__(self, name: str):
        self.name = name
        self.handled = 0
        self.total_work = 0.0

    def handle(self, req: Request) -> float:
        """Pretend to process `req`. Returns how long it took (seconds)."""
        # We don't actually sleep — it would make the notebook slow.
        # We just add the work to our counters.
        self.handled += 1
        self.total_work += req.work_seconds
        return req.work_seconds

    def __repr__(self) -> str:
        return f"Backend({self.name}, handled={self.handled}, load={self.total_work:.2f}s)"


# Quick sanity check
b = Backend("demo")
b.handle(Request(id=1, work_seconds=0.05))
print(b)


## 😬 The BAD approach: one server handles everything

Let's pretend we have **one** server and we throw 1,000 requests at it. Each
request takes a random amount of work between 10 ms and 200 ms.

We'll measure the **total time** to process all requests (sum of work) — this
is effectively how long the single server has to stay busy. If the total is
30 seconds, your users at the back of the queue wait 30 seconds.


In [ ]:
random.seed(42)  # make results reproducible

# Build a realistic workload: most requests small, a few large ones.
workload = [
    Request(id=i, work_seconds=random.uniform(0.01, 0.20))
    for i in range(1000)
]

# BAD: one lonely server
solo = Backend("solo")
for req in workload:
    solo.handle(req)

print(f"Single server handled {solo.handled} requests")
print(f"Total work it had to do: {solo.total_work:.2f} seconds")
print(f"If it handles requests one-at-a-time, the last user waits ~{solo.total_work:.1f}s 😱")


### Why this is bad

1. **Long tail latency.** The last few requests wait behind everyone in front
   of them.
2. **Single point of failure.** If this one server crashes, the whole site
   goes down.
3. **No room to grow.** The only way to handle more traffic is to buy a
   bigger machine ("vertical scaling"), which gets expensive fast and has a
   hard ceiling.

Let's fix all three by adding more backends and a load balancer in front.


## 🙂 The BETTER approach: round robin across N backends

**Round robin** is the simplest load-balancing algorithm:

> Keep a list of backends. Send request 1 to backend 0, request 2 to backend
> 1, request 3 to backend 2, … wrap around, repeat forever.

Think of dealing cards around a table: every player gets one card in turn,
then you start again from the first player.

**Pros:** tiny amount of code, perfectly fair *by request count*, predictable.
**Cons:** doesn't know anything about *how hard* each request is, or *how
busy* each backend already is. A backend that got unlucky and received three
slow requests in a row is still handed more work.


In [ ]:
class RoundRobinLoadBalancer:
    """Send each new request to the next backend in the list."""

    def __init__(self, backends: list[Backend]):
        self.backends = backends
        # `next_index` is the backend that will receive the NEXT request.
        self.next_index = 0

    def route(self, req: Request) -> Backend:
        backend = self.backends[self.next_index]
        # Advance the pointer, wrapping around to 0 at the end of the list.
        self.next_index = (self.next_index + 1) % len(self.backends)
        backend.handle(req)
        return backend


# Spin up 4 fresh backends and a round-robin LB in front of them.
backends = [Backend(f"server-{i}") for i in range(4)]
lb = RoundRobinLoadBalancer(backends)

# Replay the SAME workload so the comparison is fair.
for req in workload:
    lb.route(req)

for b in backends:
    print(b)

total = sum(b.total_work for b in backends)
worst = max(b.total_work for b in backends)
print(f"\nTotal work across the fleet: {total:.2f}s (same as before — we didn't magic away work)")
print(f"Busiest backend is busy for:  {worst:.2f}s  <-- THIS is what users actually feel")

# Round robin's actual guarantee: equal *request counts*. Assert it.
assert len({b.handled for b in backends}) == 1, [b.handled for b in backends]
assert sum(b.handled for b in backends) == len(workload)

# What it does NOT guarantee: equal *work*. The busiest backend is close to, but
# not exactly, a quarter of the total — because request sizes are random.
fair_share = total / len(backends)
print(f"Perfectly fair share would be:  {fair_share:.2f}s")
print(f"Busiest backend actually got:   {worst:.2f}s  ({worst/fair_share:.1%} of fair share)")
assert 1.0 <= worst / fair_share < 1.10, worst / fair_share

### What changed?

The **total amount of work** is exactly the same — load balancing doesn't
make requests cheaper. What changes is that the work is now spread across
4 backends running in parallel, so the *busiest* backend finishes in roughly
**¼ of the time**.

That "busiest backend finish time" is the number users feel. The wall-clock
wait for the slowest user dropped dramatically, just by adding backends and
a trivial rotation rule.


## 📊 Visualizing the load distribution

Round robin gives each backend the same **number** of requests. But because
request *sizes* are random, the total *work* per backend is only roughly
equal. Let's chart it.


In [ ]:
import matplotlib.pyplot as plt

names = [b.name for b in backends]
counts = [b.handled for b in backends]
loads = [b.total_work for b in backends]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.bar(names, counts, color="#4C9AFF")
ax1.set_title("Requests handled per backend")
ax1.set_ylabel("# requests")

ax2.bar(names, loads, color="#36B37E")
ax2.set_title("Total work per backend (seconds)")
ax2.set_ylabel("seconds of work")

plt.tight_layout()
plt.show()


## 🧠 Key takeaways

- A **single server** is both a bottleneck and a single point of failure.
- **Horizontal scaling** (more servers) needs a **load balancer** in front to
  spread work.
- **Round robin** is the simplest useful algorithm: cycle through the list.
- Round robin balances **request count** perfectly, but **total load** only
  approximately — because not all requests are equally expensive.

👉 Next: in **Notebook 2** we'll try smarter algorithms (weighted round
robin, least-connections, random) and see which one wins when request sizes
are very uneven.
